[![img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# 🦠 COVID-19 México — Demo: Pandas · Seaborn

Datos diarios de casos confirmados por estado (Feb 2020 – Jun 2023).  

Fuente: CONAHCYT -CentroGeo - GeoInt- DataLab
https://datos.covid-19.conacyt.mx/#DownZCSV

## Introducción

Este notebook es un caso de uso real de análisis de datos tabulares con Python. A partir de registros oficiales de la pandemia de COVID-19 en México se construye un flujo de trabajo completo: desde la carga del archivo crudo hasta la visualización de tendencias epidémicas por entidad federativa.

### ¿Qué se analiza?

El dataset contiene los **casos diarios confirmados de COVID-19** desagregados por estado, para el período **febrero 2020 – junio 2023**. Los datos fueron publicados por la plataforma CONAHCYT-CentroGeo-GeoInt-DataLab y reflejan los reportes agregados de hospitales, clínicas y laboratorios de todo el país.

### Herramientas utilizadas

| Biblioteca | Rol en este notebook |
|----------|----------------------|
| **Pandas** | Carga, limpieza, transformación y análisis de series temporales |
| **Seaborn** | Visualización estadística declarativa |
| **Matplotlib** | Control fino de figuras y ejes |

### Estructura del notebook

| Sección | Contenido |
|---------|-----------|
| **1. Pandas — carga y preparación** | Lectura del CSV, transposición del DataFrame, conversión al índice temporal |
| **2. Análisis nacional** | Tendencia diaria, media móvil de 7 días, patrón semanal de reporte |
| **3. Análisis estatal** | Top 6 por acumulado total, curvas por estado en formato tidy con `melt()` |

### Conceptos técnicos que se trabajan

- `DatetimeIndex` y operaciones de ventana temporal (`rolling`, `resample`)
- Formato *wide* vs. formato *tidy* y la transformación con `melt()`
- `groupby` para agregaciones por categoría
- `FacetGrid` de Seaborn para visualización de múltiples series en paneles

### Nota sobre calidad del dato

Los registros presentan un **ciclo artefactual semanal** —caídas los fines de semana y rebotes los lunes— producido por el ritmo de reporte de las fuentes, no por la dinámica real de la epidemia. Este efecto se analiza, se corrobora y se mitiga mediante una media móvil de 7 días. Es un ejemplo concreto de que los datos no siempre reflejan la realidad directamente: el análisis crítico de la fuente es parte del trabajo.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from IPython.display import display

# Suprimir advertencias menores para mantener la salida limpia
warnings.filterwarnings('ignore')

# Rutas del dataset (relativas al directorio del notebook)
DIRECTORY = 'data'
FILE = 'Casos_Diarios_Estado_Nacional_Confirmados_20230625.csv'
PATH = os.path.join(DIRECTORY, FILE)

# Tema global para todas las gráficas de Seaborn
sns.set_theme(style='darkgrid', palette='tab10')

print('Librerías listas ✔')

---
## 🐼 PARTE 1 — pandas
### 1.1 Carga y vistazo inicial

`pd.read_csv()` lee el archivo CSV y construye un `DataFrame`. Sin parámetros adicionales, Pandas infiere los tipos de datos y usa un índice entero por defecto.

Observa la **orientación original** del dataset: cada **fila** es una entidad federativa y cada **columna** es una fecha. Esta disposición (*wide format*) es habitual en archivos de distribución pública, pero no es la más conveniente para análisis con Pandas —se abordará la transformación en el paso siguiente.

In [ ]:
# pd.read_csv() infiere tipos de datos automáticamente.
# Las fechas quedarán como strings en las cabeceras de columna;
# se convertirán más adelante.
datos_crudos = pd.read_csv(PATH)
datos_crudos.head()

### 1.2 Limpieza y transformación

El dataset original tiene una **forma transpuesta** respecto a lo que Pandas espera para series temporales:

- Las **filas** son entidades federativas.
- Las **columnas** son fechas.

Para aprovechar el índice temporal de Pandas es necesario **transponer** la tabla, de modo que:

- El **índice** (`DatetimeIndex`) represente cada día de registro.
- Las **columnas** representen cada entidad federativa.

En este paso también se eliminan las columnas `'cve_ent'` y `'poblacion'`. La segunda merece una observación crítica:

#### Crítica a la columna `'poblacion'`

Los valores de esta columna corresponden al **Censo de Población y Vivienda 2020 (INEGI)**, levantado en marzo de 2020 —precisamente cuando comenzaba la pandemia en México. Se trata de una fotografía estática de la población en ese momento.

El problema de fondo es que uno de los efectos más catastróficos de la pandemia fue una **drástica sobremortalidad**: México registró entre 2020 y 2023 más de 700 000 muertes en exceso estimadas, según diversas fuentes epidemiológicas. Usar como denominador una cifra de población fija —y además obtenida antes de que se desarrollara la crisis— hace que cualquier indicador de incidencia calculado con ese dato (e.g. *casos por 100 000 habitantes*) **subestime progresivamente la tasa real** conforme avanza la serie temporal, ya que la población real era cada vez menor.

> El sitio de origen del dataset (CONAHCYT-CentroGeo-GeoInt-DataLab) **no especifica la fuente ni el año de referencia** del dato de población, lo cual es una omisión metodológica relevante para cualquier análisis de incidencia.

Por estas razones, la columna se descarta en este análisis y no se usa como denominador.

In [ ]:
# .drop() elimina las columnas de metadatos que no son conteos de casos.
# .set_index('nombre') convierte los nombres de entidad en el índice de filas.
# .transpose() invierte filas y columnas: las fechas pasan a ser el índice
#              y las entidades federativas pasan a ser las columnas.
datos_traspuestos = (datos_crudos.drop(columns=['cve_ent', 'poblacion'])
                     .set_index('nombre')
                     .transpose())
datos_traspuestos.index.name = 'fecha'
datos_traspuestos.head()

In [ ]:
# .copy() evita modificar el DataFrame original (buena práctica defensiva).
# .astype(int) convierte los conteos de float64 a int,
#   más apropiado para valores discretos.
# pd.to_datetime() transforma el índice de strings 'dd-mm-yyyy' a DatetimeIndex,
#   lo cual habilita operaciones de series temporales como .resample() y .rolling().
datos_formateados = datos_traspuestos.copy().astype(int)
datos_formateados.index = pd.to_datetime(
    datos_formateados.index, format='%d-%m-%Y')
display(datos_formateados.dtypes)
datos_formateados.head()

### 1.2.1 Datos nacionales vs. por entidad

Se dividen los datos en **dos DataFrames** con propósitos distintos:

| DataFrame | Contenido | Uso posterior |
|-----------|-----------|---------------|
| `nacional` | Una columna: total diario para todo México | Curva epidémica nacional, media móvil, análisis por día de la semana |
| `entidades` | 32 columnas, una por estado | Top 10, resampleo mensual, gráficas comparativas |

Esta separación evita que el agregado nacional —que ya es la suma de todos los estados— distorsione cálculos como totales acumulados o heatmaps.

In [ ]:
# Se selecciona la columna 'Nacional' y se convierte a DataFrame de una sola columna
# con .to_frame(), conservando el DatetimeIndex para operaciones temporales.
nacional = datos_formateados['Nacional'].to_frame()
nacional.head()

In [ ]:
# Se elimina 'Nacional' para trabajar únicamente con los 32 estados.
# Incluirla en sumas por estado produciría doble conteo,
# ya que es la suma de todas ellas.
entidades = datos_formateados.drop(columns=['Nacional'])
entidades.head()

---
## 📊 PARTE 2 — Analítica

Con los datos limpios y estructurados se aplican técnicas propias de series temporales y agregaciones: media móvil, agrupación por categoría, ranking acumulado y resampleo por frecuencia temporal.

### 2.1 Media móvil de 7 días para casos nacionales

Al observar la curva de casos diarios se aprecia un patrón llamativo: la serie presenta **valles y picos que se repiten semanalmente**. La hipótesis es que las fuentes de datos —hospitales, clínicas y laboratorios de todo el país— enviaban menos información los fines de semana, y que los reportes rezagados se acumulaban los lunes, produciendo caídas artificiales en sábado y domingo seguidas de un repunte el inicio de semana.

> Esta hipótesis será **corroborada analítica y gráficamente** en la sección 2.2, donde se calcula el promedio de casos por día de la semana y se visualiza su distribución.

Para extraer la tendencia epidemiológica real —independiente del ciclo de reporte— se aplica una **media móvil de 7 días**: al promediar exactamente una semana completa, el efecto cíclico se cancela y la curva resultante refleja únicamente la dinámica de la epidemia.

`DataFrame.rolling(window='7d')` define una ventana deslizante sobre el `DatetimeIndex`. Al pasarle una cadena de tiempo en lugar de un entero (`window=7`), Pandas calcula la ventana en **días calendario**, lo cual es correcto cuando el índice puede tener frecuencia irregular. `.mean()` promedia los valores dentro de cada ventana.

In [ ]:
# window='7d' especifica una ventana temporal de 7 días de ancho.
# .mean() calcula el promedio dentro de cada ventana.
# Nota: los primeros 6 registros tendrán promedio sobre menos de 7 observaciones
#       porque la ventana inicial está incompleta (comportamiento esperado).
nacional_media_movil_7d = (nacional
                           .rolling(window='7d')
                           .mean())
nacional_media_movil_7d.head(20)

### 2.2 Distribución de casos por día de la semana — corroboración de la hipótesis

En la sección anterior se planteó que los valles y picos semanales observados en la curva diaria se
deben al ciclo de reporte de las fuentes: **menor actividad de envío los fines de semana,
acumulación los lunes**. Esta sección corrobora esa hipótesis de dos formas:

- **Analíticamente:** calculando el promedio de casos reportados para cada día de la semana a lo
  largo de todo el período.
- **Gráficamente:** visualizando esos promedios en un barplot y su peso relativo en un pie chart.

Si la hipótesis es correcta, se esperaría ver **sábado y domingo con los promedios más bajos** y
**lunes o martes con los más altos**.

Para ello se extrae el nombre del día de cada fecha con `DatetimeIndex.to_series().dt.day_name()`,
se agrupa por día con `groupby()` y se calcula la media.

> **Nota sobre idioma:** `dt.day_name()` devuelve los nombres en inglés independientemente del
> sistema operativo. Para obtenerlos en español se puede pasar el argumento
> `locale='es_ES'` en sistemas Linux/macOS: `dt.day_name(locale='es_ES')`.

In [ ]:
# Se crea una copia para no alterar el DataFrame 'nacional'.
# .index.to_series() convierte el DatetimeIndex a una Serie para poder aplicar .dt.
# .dt.day_name() extrae el nombre del día de la semana como string (en inglés).
nacional_dias = nacional.copy()
nacional_dias['dia'] = (nacional_dias
                        .index
                        .to_series()
                        .dt
                        .day_name())
nacional_dias.head()

In [ ]:
# groupby('dia') agrupa todos los registros del mismo día de la semana.
# .mean() calcula el promedio de casos en cada grupo.
# .round(2) limita decimales para una presentación más legible.
# .sort_values() ordena de mayor a menor para identificar los días de mayor reporte.
dist_dias_nacional = (nacional_dias
                      .groupby('dia')
                      .mean()
                      .round(2)
                      .sort_values('Nacional', ascending=False))
dist_dias_nacional

### 2.3 Top 10 estados con más casos confirmados

Se suman todos los casos por columna (cada columna = un estado) para obtener el **total acumulado** durante todo el período (Feb 2020 – Jun 2023). Los 10 valores más altos identifican las entidades con mayor carga.

`DataFrame.sum()` sin argumentos opera sobre el eje 0 (columnas) y devuelve una **Serie** con el total acumulado por estado. Esta Serie se usa directamente en la gráfica de barras y como selector de columnas en el heatmap.

In [ ]:
# .sum() totaliza los casos de cada estado durante todo el período.
# .sort_values(ascending=False) ordena de mayor a menor carga acumulada.
# .head(10) retiene únicamente los 10 estados con más casos.
top_estados = (entidades
               .sum()
               .sort_values(ascending=False)
               .head(10))
top_estados

### 2.4 Casos acumulados mensuales por estado

`DataFrame.resample('ME')` es el equivalente temporal de `groupby()`: **agrupa las filas por mes** y permite aplicar funciones de agregación sobre cada grupo.

Con la frecuencia `'ME'` (*Month End*), cada grupo contiene todos los días del mes y `.sum()` produce el total mensual de casos por estado. El índice resultante tiene una entrada por mes (último día del mes, convención de Pandas para `'ME'`).

> **Nota de versión:** el alias `'M'` fue deprecado en Pandas 2.2 y eliminado en versiones posteriores. El alias correcto desde Pandas 2.2 es `'ME'` (*Month End*). Otros alias actualizados de uso frecuente: `'YE'` (*Year End*, antes `'Y'`/`'A'`), `'QE'` (*Quarter End*, antes `'Q'`).

> **Requisito:** el DataFrame debe tener un `DatetimeIndex` para poder usar `resample()` —de ahí la conversión realizada en el paso 1.2.

In [ ]:
# resample('ME') agrupa los registros diarios por mes calendario (Month End).
# .sum() suma los casos de cada mes → total mensual de casos por estado.
# El índice resultante contiene el último día de cada mes (e.g. 2020-02-29).
acumulados_mensuales = entidades.resample('ME').sum()
acumulados_mensuales.head()

---
## 🎨 PARTE 3 — Visualización con Seaborn y Matplotlib

Las visualizaciones combinan la API de alto nivel de **Seaborn** (gráficas estadísticas declarativas) con la API de bajo nivel de **Matplotlib** (control fino de ejes, títulos y disposición). Esta integración es posible porque Seaborn está construido sobre Matplotlib y comparte el mismo sistema de `Figure` y `Axes`.

Las gráficas se organizan de lo general a lo particular:

1. Curva epidémica nacional con tendencia suavizada.
2. Distribución de casos por día de la semana.
3. Ranking de los 10 estados más afectados.
4. Mapa de calor mensual para detectar olas por estado.
5. Curvas individuales de los 6 estados con más casos.

### 3.1 Curva nacional con media móvil

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# Línea delgada para los casos diarios: muestra el detalle pero con ruido visible
sns.lineplot(data=nacional,
             x=nacional.index,
             y='Nacional',
             color='steelblue',
             linewidth=1.5,
             label='Casos diarios',
             ax=ax)

# Línea gruesa para la media móvil: revela la tendencia suavizada a 7 días
sns.lineplot(data=nacional_media_movil_7d,
             x=nacional_media_movil_7d.index,
             y='Nacional',
             color='green',
             linewidth=2.5,
             label='Media móvil 7d',
             ax=ax)

ax.set_title('Curva epidémica nacional de COVID-19 en México', fontsize=16)
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Número de casos', fontsize=12)
ax.legend()
plt.tight_layout()

### 3.2 Distribución de casos por día de la semana

Se presentan dos vistas complementarias del mismo fenómeno:

- **Barplot** (`sns.barplot`): muestra el promedio de casos para cada día con intervalos de confianza, facilitando la comparación directa entre días. El orden se fija explícitamente de lunes a domingo.
- **Pie chart** (`ax.pie`): muestra la proporción relativa de cada día sobre el total, útil para comunicar el peso de fines de semana vs. días hábiles en términos porcentuales.

Ambas gráficas se colocan en una sola figura con `fig.add_subplot(2, 1, n)`, donde `2, 1` define una cuadrícula de 2 filas y 1 columna.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))

# --- Subplot 1: promedio de casos por día de la semana ---
ax1 = fig.add_subplot(2, 1, 1)
sns.barplot(data=nacional_dias,
            x='dia',
            y='Nacional',
            # El orden explícito evita que Seaborn ordene alfabéticamente
            order=['Monday',
                   'Tuesday',
                   'Wednesday',
                   'Thursday',
                   'Friday',
                   'Saturday',
                   'Sunday'],
            palette='tab10',
            ax=ax1)
ax1.set_title('Promedio de casos por día de la semana',
              fontsize=14)
ax1.set_xlabel('Día de la semana', fontsize=12)
ax1.set_ylabel('Número promedio de casos', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# --- Subplot 2: distribución porcentual (pie) ---
ax2 = fig.add_subplot(2, 1, 2)
ax2.pie(dist_dias_nacional['Nacional'],
        labels=dist_dias_nacional.index,
        autopct='%1.1f%%',           # Etiqueta de porcentaje con 1 decimal
        colors=sns.color_palette('tab10', n_colors=7))
ax2.set_title('Distribución porcentual por día de la semana', fontsize=14)

plt.tight_layout()

### 3.3 Barplot horizontal — top 10 estados

Una gráfica de barras **horizontales** es preferible a la vertical cuando los nombres de las categorías son largos (nombres de estados), ya que evita rotar etiquetas y facilita la lectura.

Se usa `Series.plot(kind='barh')` directamente sobre `top_estados`, que ya está ordenada de mayor a menor. `plt.gca().invert_yaxis()` invierte el eje Y para que el estado con más casos aparezca **en la parte superior**, siguiendo la convención de rankings descendentes.

In [ ]:
# Series.plot(kind='barh') genera barras horizontales directamente desde la Serie.
# La Serie ya está ordenada de mayor a menor; invert_yaxis() pone el #1 arriba.
top_estados.plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Top 10 Estados con más casos confirmados de COVID-19', fontsize=16)
plt.xlabel('Número total de casos (millones)', fontsize=12)
plt.ylabel('Estado', fontsize=12)
plt.gca().invert_yaxis()
plt.tight_layout()

### 3.4 Mapa de calor mensual — top 10 estados

`sns.heatmap()` codifica valores numéricos como **intensidad de color**, permitiendo identificar de un vistazo patrones temporales que serían difíciles de leer en una tabla o en 10 líneas superpuestas.

- **Eje X:** meses del período (Feb 2020 – Jun 2023).
- **Eje Y:** los 10 estados con más casos totales.
- **Color (`'Reds'`):** cantidad de casos confirmados; rojo oscuro = más casos.

Se transpone el DataFrame (`acumulados_mensuales[...].T`) para que los **estados queden en filas** y los **meses en columnas**, orientación convencional para este tipo de heatmap.

In [ ]:
# Etiquetas de mes en formato 'aa-mm' para el eje X
# (más compactas que la fecha completa).
meses = acumulados_mensuales.index.strftime('%y-%m')

fig, ax = plt.subplots(figsize=(12, 8))

# Se filtra por top_estados.index para mostrar sólo los 10 estados de interés.
# .T transpone: estados en filas (eje Y), meses en columnas (eje X).
sns.heatmap(acumulados_mensuales[top_estados.index].T,
            cmap='Reds',
            # Líneas delgadas entre celdas para mejorar legibilidad
            linewidths=0.5,
            linecolor='gray',
            cbar_kws={'label': 'Número de casos'})

ax.set_title('Calor mensual de casos confirmados por estado (Top 10)', fontsize=16)
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Estado', fontsize=12)
ax.set_xticklabels(meses, rotation=45, ha='right', fontsize=8)
plt.tight_layout()

### 3.5 Curvas epidémicas por estado — Top 6

#### El formato *tidy* (datos ordenados)

El concepto de **tidy data** fue formalizado por Hadley Wickham (2014) y establece tres reglas que todo dataset "ordenado" debe cumplir:

| Regla | Significado |
|-------|-------------|
| **1.** Cada *variable* ocupa exactamente una columna | Una columna = un atributo medido |
| **2.** Cada *observación* ocupa exactamente una fila | Una fila = un evento o medición individual |
| **3.** Cada *unidad observacional* forma su propia tabla | No mezclar granos distintos en el mismo DataFrame |

---

**¿Por qué el DataFrame `entidades` *no* está en formato tidy?**

En `entidades` cada columna es un estado diferente —es decir, una misma variable (*número de casos*) está repartida en 32 columnas. Esto se llama **formato ancho** (*wide*):

```
fecha       | Ciudad de México | Estado de México | Jalisco | ...
------------|-----------------|-----------------|---------|----
2020-03-01  |        5        |        2        |    1    | ...
2020-03-02  |        8        |        3        |    4    | ...
```

El nombre del estado es en realidad **una variable** (`Estado`), y su valor debería estar en una columna propia. El formato correcto —**tidy** o *long*— tiene una fila por cada combinación fecha × estado:

```
fecha       | Estado           | Casos
------------|-----------------|------
2020-03-01  | Ciudad de México |   5
2020-03-01  | Estado de México |   2
2020-03-01  | Jalisco          |   1
2020-03-02  | Ciudad de México |   8
...
```

---

**¿Por qué importa?**

La mayoría de las herramientas de análisis y visualización —Seaborn, Plotnine, `groupby`, `pivot_table`— esperan datos en formato tidy porque pueden mapear directamente **columnas a dimensiones**:

- `hue='Estado'` → colorea por estado
- `col='Estado'` → crea un panel por estado
- `groupby('Estado').sum()` → agrega por estado

Con el formato ancho estas operaciones requieren código ad hoc por cada estado, lo que no escala.

---

**La transformación con `DataFrame.melt()`**

`melt()` es la operación que convierte de formato ancho a tidy:

- `id_vars` — columnas que *no* se transforman (actúan como identificadores de la observación).
- `var_name` — nombre que se le dará a la nueva columna que contendrá los encabezados originales.
- `value_name` — nombre que se le dará a la nueva columna que contendrá los valores.

La operación inversa —de tidy a wide— se hace con `DataFrame.pivot()` o `DataFrame.pivot_table()`.

In [ ]:
# .head(6) selecciona los 6 estados con más casos (top_estados ya está ordenado).
# .index.tolist() extrae los nombres de estado como lista de strings.
top_6_cols = top_estados.head(6).index.tolist()

# .reset_index() saca la fecha del índice para que melt pueda tratarla como columna.
# .melt() convierte de formato ancho a formato largo (tidy):
#   - id_vars='fecha'    → columna que actúa como identificador (no se "derrite")
#   - var_name='Estado'  → nombre de la nueva columna con los nombres de estados
#   - value_name='Casos' → nombre de la nueva columna con los conteos
top_6_estados = (entidades[top_6_cols]
                 .reset_index()
                 .melt(id_vars='fecha',
                       var_name='Estado',
                       value_name='Casos'))
top_6_estados.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# hue='Estado' asigna automáticamente un color distinto a cada estado.
# Todas las series comparten la misma escala en Y, lo que facilita comparar magnitudes.
sns.lineplot(data=top_6_estados,
             x='fecha',
             y='Casos',
             hue='Estado',
             palette='tab10',
             linewidth=1.5,
             ax=ax)

ax.set_title(
    'Curvas epidémicas de los 6 estados con más casos confirmados',
    fontsize=16)
ax.set_xlabel('Fecha', fontsize=12)
ax.set_ylabel('Número de casos', fontsize=12)
ax.legend(title='Estado', fontsize=10)
plt.tight_layout()

#### FacetGrid — pequeños múltiplos por estado

`sns.FacetGrid` crea una cuadrícula de subplots donde cada panel muestra los datos de un subconjunto (`col='Estado'`). A diferencia del gráfico anterior donde todos los estados comparten el eje Y, aquí `sharey=False` permite que cada panel use su propia escala, lo que facilita ver la **forma de la curva** de estados con pocos casos que quedarían aplastados en una escala compartida.

In [ ]:
fig = plt.figure(figsize=(14, 12))

# FacetGrid crea la cuadrícula: col='Estado' genera un panel por estado,
# col_wrap=3 limita a 3 columnas por fila,
# sharey=False escala Y de forma independiente por panel.
(sns.FacetGrid(top_6_estados,
               col='Estado',
               col_wrap=3,
               height=4,
               sharey=False)
    .map(sns.lineplot,
         'fecha',
         'Casos',
         linewidth=1.5)      # Aplica lineplot a cada panel
    .set_titles('{col_name}')  # Título de cada panel = nombre del estado
    .set_axis_labels('Fecha', 'Número de casos')
    .set_xticklabels(fontsize=8))

plt.subplots_adjust(top=0.9)
plt.suptitle('Curvas epidémicas individuales por estado (Top 6)', fontsize=16)
plt.tight_layout()

---
## 📋 Conclusiones

### Dinámica epidémica nacional

La curva suavizada (media móvil 7d) permite distinguir **tres olas principales**:

| Ola | Período aproximado | Característica |
|-----|--------------------|----------------|
| **Primera** | May – Sep 2020 | Crecimiento exponencial inicial; capacidad hospitalaria al límite |
| **Segunda** | Nov 2020 – Feb 2021 | Pico más alto en mortalidad; variante Alfa/Beta dominante |
| **Tercera** | Dic 2021 – Feb 2022 | Mayor volumen de casos confirmados (variante Ómicron), menor mortalidad relativa |

A partir de mediados de 2022 la intensidad del reporte decayó de forma progresiva, lo que puede
reflejar tanto una reducción real de la transmisión como una menor cobertura del sistema de
vigilancia.

### Patrón de reporte semanal

El análisis por día de la semana **corroboró la hipótesis**: lunes y martes concentran los promedios
más altos de casos reportados, mientras que sábado y domingo presentan los valores más bajos. Este
patrón es consistente a lo largo de todo el período y confirma que las caídas y picos semanales en
la curva diaria son artefactos del ciclo de reporte, no de la dinámica epidemiológica real. La media
móvil de 7 días elimina este efecto de forma eficaz.

### Distribución geográfica

- **Ciudad de México, Estado de México y Jalisco** concentraron la mayor carga acumulada de casos,
  coherente con su peso demográfico y densidad urbana.
- El mapa de calor mensual muestra que las olas no afectaron a todos los estados simultáneamente
  ni con la misma intensidad: algunos estados del norte experimentaron picos más tempranos en la
  primera ola, mientras que los estados del centro dominaron en la segunda.

### Limitaciones del dataset

- **Columna `poblacion`:** corresponde al Censo de Población y Vivienda 2020 (INEGI) y no
  refleja la sobremortalidad acumulada durante la pandemia (estimada en más de 700 000 muertes
  en exceso). Usarla como denominador produciría tasas de incidencia cada vez más imprecisas
  conforme avanza la serie temporal.
- **Documentación de la fuente:** el sitio de origen no especifica la fuente del dato de
  población ni el método de estimación de casos confirmados, lo que limita la reproducibilidad
  y la trazabilidad del análisis.
- **Subregistro:** los casos confirmados dependen de la capacidad de prueba y reporte del
  sistema de salud; en períodos de alta demanda, el subregistro pudo ser significativo.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>

<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>
